## Prompt Engineering in LangSmith with Gemini

### Import environment variables

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

### Pull in Prompt from Prompthub

In [2]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate
from langsmith.utils import LangSmithNotFoundError

client = Client(api_key=LANGSMITH_API_KEY)

# Define the prompt template
eli5_prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. 

Your task is to take a complex question and context information, then provide a clear, simple explanation using:
- Simple words and concepts
- Analogies and examples from everyday life
- Short sentences
- Engaging and friendly tone

Keep your explanation concise but complete.

Question: {question}

Context: {context}

Please explain this in simple terms that a 5-year-old would understand:
""")])

# Try to pull the prompt, if it doesn't exist, push it first
try:
    print("Trying to pull existing prompt...")
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled existing prompt from LangSmith")
except LangSmithNotFoundError:
    print("❌ Prompt not found. Creating and pushing new prompt...")
    
    # Push the prompt to LangSmith
    client.push_prompt(
        "eli5-concise",
        object=eli5_prompt_template,
        description="A prompt for explaining complex topics in simple terms that a 5-year-old could understand"
    )
    print("✅ Successfully pushed prompt to LangSmith")
    
    # Now pull the prompt back
    prompt = client.pull_prompt("eli5-concise", include_model=True)
    print("✅ Successfully pulled the newly created prompt")

Trying to pull existing prompt...
✅ Successfully pulled existing prompt from LangSmith


### Setup Gemini AI Application

Let's first setup our web search tool, as usual

In [3]:
# Initialize a web-search tool if you have Tavily configured. Otherwise, this falls back to a simple context string.

try:
    from langchain_tavily import TavilySearch
    web_search_tool = TavilySearch(max_results=1)
except Exception:
    web_search_tool = None


Let's now create our Gemini application, same as in the tracing module. This time, our prompt is the one pulled from PromptHub

In [26]:
import os
from google import genai
from langsmith import traceable

# Create Gemini application
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
chat = client.chats.create(model="gemini-3.6-flash")

@traceable
def search(question):
    return ["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])]
    
@traceable
def explain(question, context):
    # Use the LangSmith-pulled prompt as a formatted string
    formatted = prompt.format(question=question, context=context)
    response = chat.send_message(formatted)
    return response.text

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer


### Test Gemini Application

In [28]:
question = "what is complexity economics?"
context = search(question)
context

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [29]:
formatted = prompt.format(question=question, context=context)
formatted

"System: You are an expert at explaining complex topics in simple terms that a 5-year-old could understand. \n\nYour task is to take a complex question and context information, then provide a clear, simple explanation using:\n- Simple words and concepts\n- Analogies and examples from everyday life\n- Short sentences\n- Engaging and friendly tone\n\nKeep your explanation concise but complete.\n\nQuestion: what is complexity economics?\n\nContext: ['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation t

In [24]:
question = "what is complexity economics?"
print(eli5(question))

Imagine the world’s money and shops are like a giant playground full of kids!

A long time ago, some grown-ups thought the economy worked like a simple toy train. They thought it always went around the same track, doing the exact same thing, very predictably.

But **complexity economics** says: "No way! It’s actually like a huge game of tag!"

In a game of tag:
* Everyone makes their own choices.
* You change what you do based on what your friends do. 
* If someone runs fast, you have to dodge! 
* Everyone learns new tricks as they play.

Because everyone is reacting to each other, the game is always changing and full of surprises. 

So, **complexity economics** is just studying how people and money work together like a lively, ever-changing playground instead of a boring robot machine!


In [18]:
question = "what is complexity economics?"
["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])]

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [17]:
[d["content"] for d in web_search_tool.invoke({"query": question})["results"]]

['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb

In [21]:
if web_search_tool is not None:
    try:
        print(["\n".join([d["content"] for d in web_search_tool.invoke({"query": question})["results"]])])
    except Exception:
        print("Error")


['Complexity economics, or economic complexity, is the application of complexity science to the problems of economics. It relaxes several common assumptions in economics, including general equilibrium theory. While it does not reject the existence of an equilibrium, it features a non-equilibrium approach and sees such equilibria as a special case and as an emergent property resulting from complex interactions between economic agents. The complexity science approach has also been applied as the [...] Complexity economics has a complex relation to previous work in economics and other sciences, and to contemporary economics. Complexity-theoretic thinking to understand economic problems has been present since their inception as academic disciplines. Research has shown that no two separate micro-events are completely isolated, and there is a relationship that forms a macroeconomic structure. However, the relationship is not always in one direction; there is a reciprocal influence when feedb